# GristMill — Model Comparison

This notebook compares two trained ONNX sieve classifiers (candidate vs baseline)
using benchmark texts and produces a `ComparisonReport` with accuracy, agreement
rate, latency percentiles, and label distribution charts.

**Prerequisites:** `pip install -e gristmill-ml onnxruntime sentence-transformers matplotlib`

In [ ]:
import sys
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path('..') / 'src'))

MODELS_DIR = Path('~/.gristmill/models').expanduser()

print(f'Models directory: {MODELS_DIR}')
if MODELS_DIR.exists():
    onnx_files = list(MODELS_DIR.glob('*.onnx'))
    print(f'ONNX models found: {[f.name for f in onnx_files]}')
else:
    print('Models directory does not exist — will use dummy models for demo.')

In [ ]:
from gristmill_ml.experiments.comparisons import ModelInfo

# Adjust these paths to your actual model files.
# If you have trained a model with the walkthrough notebook it will be at:
CANDIDATE_PATH = MODELS_DIR / 'intent-classifier-v1.onnx'
BASELINE_PATH  = MODELS_DIR / 'intent-classifier-v1-walkthrough.onnx'

# Fall back gracefully if one of the models doesn't exist
if not CANDIDATE_PATH.exists() and not BASELINE_PATH.exists():
    print('[demo mode] No ONNX models found — creating minimal dummy models.')
    import torch, torch.onnx
    from gristmill_ml.training.sieve_trainer import SieveClassifierHead, FEATURE_DIM
    MODELS_DIR.mkdir(parents=True, exist_ok=True)
    for p in [CANDIDATE_PATH, BASELINE_PATH]:
        m = SieveClassifierHead()
        dummy = torch.zeros(1, FEATURE_DIM)
        torch.onnx.export(m, dummy, str(p), opset_version=17,
                          input_names=['features'], output_names=['logits'],
                          dynamic_axes={'features': {0: 'batch'}, 'logits': {0: 'batch'}})
        print(f'  Created dummy: {p}')
elif not BASELINE_PATH.exists():
    print(f'Baseline not found — using candidate as baseline for demo.')
    import shutil
    shutil.copy(str(CANDIDATE_PATH), str(BASELINE_PATH))

candidate = ModelInfo(path=CANDIDATE_PATH, label='sieve-candidate')
baseline  = ModelInfo(path=BASELINE_PATH,  label='sieve-baseline')

print(f'Candidate: {candidate.label} — {candidate.path}')
print(f'Baseline : {baseline.label} — {baseline.path}')

In [ ]:
from gristmill_ml.datasets.loaders import load_mmlu, BenchmarkSample

# Load benchmark texts (with graceful fallback if datasets lib unavailable)
benchmark = load_mmlu(subject='all', split='test', n=50)

# Supplement with routing-specific texts for better coverage
routing_texts = [
    'status',
    'ping',
    'list files',
    'schedule meeting tomorrow at 3pm',
    'remind me at 09:00',
    'run daily backup',
    'summarise the last 10 support tickets with sentiment',
    'classify these 50 emails and draft replies',
    'explain why the authentication service failed intermittently last week',
    'write a blog post about Rust ownership',
]
routing_samples = [BenchmarkSample(question=t, answer='', subject='routing', split='demo') for t in routing_texts]

all_samples = benchmark + routing_samples
texts = [s.question for s in all_samples]

print(f'Total evaluation texts: {len(texts)}')
print(f'  From MMLU        : {len(benchmark)}')
print(f'  Routing-specific : {len(routing_texts)}')

In [ ]:
from gristmill_ml.experiments.comparisons import compare_models

print('Running model comparison (this may take a minute for embedding)…')
report = compare_models(
    candidate=candidate,
    baseline=baseline,
    texts=texts,
    labels=None,  # no ground-truth labels for these texts
)
print('Comparison complete.')

In [ ]:
report.print_report()

In [ ]:
try:
    import matplotlib.pyplot as plt
    import numpy as np

    labels_list = ['LOCAL_ML', 'RULES', 'HYBRID', 'LLM_NEEDED']
    cand_vals = [report.label_distribution_candidate.get(l, 0) for l in labels_list]
    base_vals = [report.label_distribution_baseline.get(l, 0) for l in labels_list]

    x = np.arange(len(labels_list))
    width = 0.35

    fig, ax = plt.subplots(figsize=(10, 5))
    bars1 = ax.bar(x - width/2, cand_vals, width, label=candidate.label, color='#2196F3', alpha=0.85)
    bars2 = ax.bar(x + width/2, base_vals, width, label=baseline.label,  color='#FF9800', alpha=0.85)

    ax.set_title('Per-class Label Distribution: Candidate vs Baseline', fontsize=13)
    ax.set_xticks(x)
    ax.set_xticklabels(labels_list)
    ax.set_ylabel('Sample count')
    ax.legend()
    ax.bar_label(bars1, padding=3)
    ax.bar_label(bars2, padding=3)
    plt.tight_layout()
    plt.show()
except ImportError:
    print('(matplotlib not installed — skipping bar chart)')
    print('Candidate distribution:', report.label_distribution_candidate)
    print('Baseline  distribution:', report.label_distribution_baseline)

In [ ]:
try:
    import matplotlib.pyplot as plt

    latency_data = {
        candidate.label: [report.candidate_latency_p50_ms, report.candidate_latency_p99_ms],
        baseline.label:  [report.baseline_latency_p50_ms,  report.baseline_latency_p99_ms],
    }

    fig, ax = plt.subplots(figsize=(7, 5))
    percentiles = ['p50', 'p99']
    x = [0, 1]
    colors = ['#2196F3', '#FF9800']

    for i, (name, vals) in enumerate(latency_data.items()):
        ax.bar([xi + i * 0.35 for xi in x], vals, 0.3, label=name, color=colors[i], alpha=0.85)

    ax.set_xticks([0.175, 1.175])
    ax.set_xticklabels(percentiles)
    ax.set_ylabel('Latency (ms per sample)')
    ax.set_title('Inference Latency Comparison (per-sample)', fontsize=13)
    ax.legend()
    plt.tight_layout()
    plt.show()
except ImportError:
    print('(matplotlib not installed — skipping latency plot)')
    print(f'Candidate: p50={report.candidate_latency_p50_ms:.2f}ms  p99={report.candidate_latency_p99_ms:.2f}ms')
    print(f'Baseline:  p50={report.baseline_latency_p50_ms:.2f}ms  p99={report.baseline_latency_p99_ms:.2f}ms')

In [ ]:
IMPROVEMENT_THRESHOLD = 0.02  # 2 percentage points

print('=' * 60)
print('Deployment Recommendation')
print('=' * 60)

if report.candidate_accuracy < 0:
    # No ground-truth labels — base recommendation on agreement and latency
    latency_improvement = report.baseline_latency_p99_ms - report.candidate_latency_p99_ms
    if latency_improvement > 1.0:
        print(f'RECOMMEND candidate: p99 latency improved by {latency_improvement:.2f}ms')
    elif report.agreement_rate >= 0.95:
        print(f'NEUTRAL: Models agree on {report.agreement_rate:.1%} of samples — consider A/B testing.')
    else:
        print(f'INVESTIGATE: Models only agree on {report.agreement_rate:.1%} of samples.')
else:
    delta = report.candidate_accuracy - report.baseline_accuracy
    if delta >= IMPROVEMENT_THRESHOLD:
        print(f'RECOMMEND candidate: accuracy improved by {delta:+.4f} ({delta * 100:+.2f}%)')
    elif delta >= 0:
        print(f'MARGINAL improvement: {delta:+.4f}. Consider runtime/latency trade-off.')
    else:
        print(f'KEEP baseline: candidate regressed by {delta:.4f}.')

print(f'\nAgreement rate: {report.agreement_rate:.4f}')
print(f'Candidate p99 latency: {report.candidate_latency_p99_ms:.2f}ms')
print(f'Baseline  p99 latency: {report.baseline_latency_p99_ms:.2f}ms')